In [1]:
%%capture output
! pip install ibllib
! pip install pynapple
! pip install git+https://github.com/int-brain-lab/paper-brain-wide-map.git
! pip install -U google-colab

In [6]:
# system
from pathlib import Path
from tqdm import tqdm
import pickle

# analysis
import numpy as np
import pandas as pd
import pynapple as nap
from scipy import stats
import glob 
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import warnings
import os

import scipy.ndimage
from iblutil.numerical import bincount2D
from iblatlas.atlas import BrainRegions
from brainbox.io.one import SpikeSortingLoader, SessionLoader


# visualization
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm
import patsy
from scipy.stats import pearsonr
from matplotlib.collections import LineCollection

from umap import UMAP
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from collections import defaultdict

from reproducible_ephys_functions import get_insertions
from one.api import ONE
one= ONE()

In [7]:
def idxs_from_files(design_matrices):
    
    idxs = []
    mouse_names = []
    for m, mat in enumerate(design_matrices):
        mouse_name = design_matrices[m][51:]
        eid = design_matrices[m][14:50]
        idx = str(eid + '_' + mouse_name)

        if len(idxs) == 0:
            idxs = idx
            mouse_names = mouse_name
        else:
            idxs = np.hstack((idxs, idx))
            mouse_names = np.hstack((mouse_names, mouse_name))
            
    return idxs, mouse_names


In [8]:
prefix = '/home/ines/repositories/'
# prefix = '/Users/ineslaranjeira/Documents/Repositories/'


# Load trial data

In [10]:
insertions = get_insertions(level=0, one=one, freeze='freeze_2024_03')
repro_ephys_insertions = pd.DataFrame.from_dict(insertions)

In [11]:
repro_eid = repro_ephys_insertions['probe_insertion']

In [12]:
import brainwidemap
# this dataframe holds a summary of all the sessions
# and for us importantly, the eids and pids
bwm_df = brainwidemap.bwm_query()  # each row of this dataframe is a recording

n_sessions = bwm_df["eid"].unique().shape[0]
n_insertions = bwm_df["pid"].unique().shape[0]
print(
    f"{n_sessions} sessions with {n_insertions} individual neuropixel recordings"
)
# bwm_df.head()
bwm_pid = bwm_df['pid'].unique()

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
459 sessions with 699 individual neuropixel recordings


## Getting units from the brainwide map

In [13]:
units_df = brainwidemap.bwm_units(one)
n_units = units_df.shape[0]
print(f"{n_units} units present in the table")
# units_df.head()

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01
62990 units present in the table


In [14]:
# cluster_df = pd.read_pickle(data_path+'extended_mouse_LDA_5_bins_cut')
data_path = prefix + 'representation_learning_variability/paper-individuality/clustering/'
# session_cluster = pd.read_parquet(data_path+'cluster_per_session')
lda = pd.read_pickle(data_path+'mouse_LDA_5_bins_cut06-07-2026')

lda_eid = lda.loc[lda['session'].isin(list(bwm_df.eid)), 'session']
lda_pid = bwm_df.loc[bwm_df['eid'].isin(lda_eid), 'pid']

In [15]:
print(len(lda_eid))
print(len(lda_pid))

244
380


In [17]:
use_eid = lda_pid
# use_eid = repro_eid
# use_eid = bwm_pid

In [18]:
len(use_eid)

380

In [19]:
repro_units = units_df.loc[units_df['pid'].isin(list(use_eid))]

## Querying and loading the data

Then we will query the brain region ACA. ACA has many sub-regions, and we need to get the list of neurons belinging to any of the sub-regions.

In [20]:
BRAIN_REGIONS = ['VISa', 'VISam', 'CA1', 'DG', 'LP', 'PO']  # Reproducible ephys regions
regions = BrainRegions()
BRAIN_REGIONS = repro_units['Beryl'].unique()  # All available in the session query

# Find relevant behavioral data

In [21]:
# LOAD DATA
data_path = prefix + 'representation_learning_variability/paper-individuality/data/design_matrices/'

all_files = os.listdir(data_path)
design_matrices = [item for item in all_files if 'design_matrix' in item and 'standardized' not in item]
idxs, mouse_names = idxs_from_files(design_matrices)

# Identify sessions available to process
sessions_to_process = []
for m, mat in enumerate(idxs):
    mouse_name = mat[37:]
    session = mat[:36]
        
    sessions_to_process.append((mouse_name, session))
print(f"Found {len(sessions_to_process)} sessions to process.")

Found 319 sessions to process.


# Process and save PETH with trial info

In [22]:
states_path = prefix + 'representation_learning_variability/paper-individuality/data/states_files/'
save_states_path = prefix + 'representation_learning_variability/paper-individuality/data/neuron_files/'

In [23]:
files = os.listdir(save_states_path)
len(files)

0

In [24]:
regions = BrainRegions()
total_files = len(list(use_eid))
for p, pid in enumerate(list(use_eid)):
    if (p + 1) % max(1, total_files // 10) == 0 or (p + 1) == total_files:
        print(f"Progress: {(p + 1) / total_files * 100:.1f}% ({p + 1}/{total_files}) files processed")
    check_filename = "states_neurons_file_" + str(pid)
    if check_filename not in files:
        ssl = SpikeSortingLoader(one=one, pid=pid)
        eid = ssl.eid
        session_files = glob.glob(os.path.join(states_path, f"*{eid}*"))
        # Process ephys sessions if they have behavior
        if session_files:
            session_file_path = session_files[0]  # Take the first (and only) match
            states_trial = pd.read_parquet(session_file_path)
            identifiable_states = states_trial["identifiable_states"]
            digits = identifiable_states.str.extract(r'(\d)(\d)(\d)').astype('Int64')
            digits.columns = ["paw", "whisk", "lick"]
            states_trial['paw'] = digits['paw']
            states_trial['whisk'] = digits['whisk']
            states_trial['lick'] = digits['lick']

            # Initialize the final dataframe with states_trial
            states_trial_with_spikes = states_trial.copy()
            # Add session identifier
            states_trial_with_spikes['session_eid'] = eid
            states_trial_with_spikes['session_pid'] = pid

            # Load spike sorting once for the whole session
            spikes, clusters, channels = ssl.load_spike_sorting(good_units=True, revision='2025-05-26')
            df_clus = pd.DataFrame(ssl.merge_clusters(spikes, clusters, channels))

            # Build the histogram bin edges ONCE, from the full behavioral grid. These must NOT be
            # recomputed inside the area loop, and rows must NOT be dropped until every neuron has
            # been binned on the same grid -- otherwise areas processed after the first would be
            # binned over a frame with the ~18% NaN-trial_id rows already removed, producing
            # gap-spanning bins and inflated/misaligned counts.
            time_bins = states_trial_with_spikes['Bin'].values
            expected_bin_width = 1/60
            # Center each timestamp with fixed width; interior edges are midpoints between bins.
            bin_edges = np.empty(len(time_bins) + 1)
            bin_edges[0] = time_bins[0] - expected_bin_width / 2
            bin_edges[1:-1] = (time_bins[:-1] + time_bins[1:]) / 2
            bin_edges[-1] = time_bins[-1] + expected_bin_width / 2

            # Collect every neuron's spike-count column here and attach in a single concat at the
            # end (assigning columns one-by-one fragments the DataFrame across hundreds of inserts).
            neuron_counts = {}

            # This is where we select the units belonging to each available area
            unique_areas = df_clus['atlas_id'].dropna().unique()
            for a, area in enumerate(unique_areas):
                area_name = regions.id2acronym(area)
                aca_leaf_nodes = regions.descendants(regions.acronym2id(area_name))
                # This is where we select the units belonging to any of the leaf nodes brain regions
                selection_clusters = df_clus['atlas_id'].isin(aca_leaf_nodes['id'])
                iclusters = np.where(selection_clusters)[0]
                ispikes = np.isin(spikes['clusters'], iclusters)
                st = spikes['times'][ispikes]
                sc = spikes['clusters'][ispikes]

                # Filter spikes to only include those within the states_trial time range
                time_mask = (st >= bin_edges[0]) & (st <= bin_edges[-1])
                st_filtered = st[time_mask]
                sc_filtered = sc[time_mask]

                cluster_ids = np.unique(sc_filtered)
                for cluster_id in cluster_ids:
                    # Process a single cluster
                    cluster_spikes = st_filtered[sc_filtered == cluster_id]
                    # Use histogram to bin spikes into the same time bins as states_trial
                    counts, _ = np.histogram(cluster_spikes, bins=bin_edges)
                    neuron_counts[f'{area_name[0]}_neuron_{int(cluster_id)}_spike_count'] = counts

            # Attach all neuron columns at once, then drop the non-trial (NaN trial_id) bins
            if neuron_counts:
                states_trial_with_spikes = pd.concat(
                    [states_trial_with_spikes,
                     pd.DataFrame(neuron_counts, index=states_trial_with_spikes.index)],
                    axis=1)
            states_trial_with_spikes = states_trial_with_spikes.dropna(subset=['trial_id'])

            # Save per session
            save_filename = save_states_path + "states_neurons_file_" + str(pid)
            with open(save_filename, 'wb') as f:
                pickle.dump(states_trial_with_spikes, f)

100%|██████████| 3/3.0 [00:01<00:00,  2.49it/s]


Progress: 10.0% (38/380) files processed


100%|██████████| 4/4.0 [00:01<00:00,  2.90it/s]
100%|██████████| 3/3.0 [00:01<00:00,  2.68it/s]
100%|██████████| 3/3.0 [00:00<00:00,  3.16it/s]


Progress: 20.0% (76/380) files processed


100%|██████████| 3/3.0 [00:01<00:00,  2.73it/s]


...

Progress: 50.0% (190/380) files processed


100%|██████████| 3/3.0 [00:01<00:00,  2.92it/s]
100%|██████████| 3/3.0 [00:00<00:00,  3.20it/s]


Progress: 60.0% (228/380) files processed


100%|██████████| 5/5.0 [00:01<00:00,  2.62it/s]


Progress: 70.0% (266/380) files processed
Progress: 80.0% (304/380) files processed
Progress: 90.0% (342/380) files processed
Progress: 100.0% (380/380) files processed
